# Grok + .faf: remote MCP context

Connect Grok (via the xAI SDK) to a remote .faf MCP server, giving a session structured, persistent project context — no local setup.

## Overview

Code and docs alone give a model only partial project context. A .faf MCP server serves the structured context — the 6Ws, stack, and commands from `project.faf` — over the wire, and keeps it in sync.

This example points at the public grok-faf-mcp endpoint. Swap `server_url` for any .faf MCP server; see the grok-faf-mcp README to run your own.

In [ ]:
import os
from dotenv import load_dotenv

from xai_sdk import Client
from xai_sdk.tools import mcp

load_dotenv()

api_key = os.getenv("XAI_API_KEY")
if not api_key:
    raise ValueError("Set XAI_API_KEY in .env or environment")

client = Client(api_key=api_key)

In [ ]:
# Register the remote .faf MCP server (public grok-faf-mcp endpoint)
tools = [
    mcp(
        server_url="https://grok-faf-mcp.vercel.app/sse",
        server_label="grok_faf_mcp",
    )
]

chat = client.chat.create(model="grok-4-1-fast", tools=tools)

print("Remote MCP tools registered:")
print([tool.function.name for tool in chat.tools])

In [ ]:
# A prompt that exercises the .faf context tools
chat.append({
    "role": "user",
    "content": "Using the .faf context, summarise this project and report its AI-readiness score.",
})

print("\nStreaming response:\n")
for response in chat.stream():
    if response.delta:
        print(response.delta, end="")

## What the tools provide

- `faf_score` — the deterministic AI-readiness score (same input, same number).
- `faf_context` — the structured project slots from `project.faf`.
- bi-sync keeps the served context current as the repo changes.

The same pattern works in any Grok session — notebooks, agents, or API builds. To serve your own project, run the server from the [grok-faf-mcp](https://github.com/Wolfe-Jam/grok-faf-mcp) repo.